# Diffusion Trajectory — Denoising with Matplotlib FuncAnimation

This notebook visualizes a **denoising diffusion trajectory** using Matplotlib's `FuncAnimation`.

Concept: starting from pure Gaussian noise, a reverse-diffusion process gradually denoises toward a target 2D Gaussian mixture. Each frame shows the marginal distribution $p_t(x)$ at timestep $t$.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets

# Generate target data: 2D Gaussian mixture
np.random.seed(42)
n_samples = 2000
mix = np.random.choice([0,1,2], size=n_samples, p=[0.4,0.35,0.25])
means = [np.array([-1.5, -1.0]), np.array([1.5, 0.5]), np.array([0.0, 1.5])]
covs = [np.eye(2)*0.3, np.eye(2)*0.25, np.eye(2)*0.2]

target = np.zeros((n_samples, 2))
for i in range(n_samples):
    target[i] = np.random.multivariate_normal(means[mix[i]], covs[mix[i]])

# Forward diffusion: x_t = sqrt(alpha_t) * x_0 + sqrt(1-alpha_t) * eps
T = 50
trajectory = []
for t in range(T+1):
    alpha = 1.0 - (t / T) * 0.99  # schedule
    noise = np.random.randn(*target.shape)
    x_t = np.sqrt(alpha) * target + np.sqrt(1-alpha) * noise
    trajectory.append(x_t)

# Reverse order so we start from noise and denoise
trajectory = trajectory[::-1]
print(f'Trajectory length: {len(trajectory)}')

In [ ]:
# Build FuncAnimation
fig, ax = plt.subplots(figsize=(6,6))
scatter = ax.scatter([], [], c='steelblue', s=10, alpha=0.6)
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.set_title('Denoising Diffusion Trajectory')
ax.set_xlabel('x')
ax.set_ylabel('y')
time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top')

def init():
    scatter.set_offsets(np.empty((0,2)))
    time_text.set_text('')
    return scatter, time_text

def update(frame):
    data = trajectory[frame]
    scatter.set_offsets(data)
    time_text.set_text(f'Step {frame}/{T}')
    return scatter, time_text

ani = FuncAnimation(fig, update, frames=len(trajectory), init_func=init, blit=True)
ani.save('diffusion_trajectory.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close(fig)
display(HTML("<video controls src='diffusion_trajectory.mp4' width=480></video>"))

## Interactive Parameter Explorer

Use the slider to jump to a specific timestep without re-rendering the animation.

In [ ]:
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt

def show_frame(t=0):
    fig, ax = plt.subplots(figsize=(5,5))
    data = trajectory[t]
    ax.scatter(data[:,0], data[:,1], c='steelblue', s=10, alpha=0.6)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_title(f'Diffusion Step {t}/{T}')
    plt.show()

interact(show_frame, t=IntSlider(min=0, max=T, step=1, value=0, description='Timestep'))

## Expected Output

- A video player showing the 2D point cloud morphing from noise to structured Gaussian mixture
- An interactive slider that lets you scrub through individual frames instantly